In [23]:
from openai import OpenAI
from pymilvus import model as milvus_model
from pymilvus import MilvusClient
from tqdm import tqdm

import os
import warnings
import logging
import re

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("pymilvus").setLevel(logging.ERROR)

# ====================== 1. 读取 API KEY ======================
api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    print("Can not find deepseek api key")
    exit()
else:
    print("✅ Api key read successfully")

# ====================== 2. 初始化大模型 ======================
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"
)

# ====================== 3. 初始化向量模型 ======================
embedding_model = milvus_model.DefaultEmbeddingFunction()
test_embedding = embedding_model.encode_queries(["Vector len"])[0]
embedding_dim = len(test_embedding)

print(f"向量维度: {embedding_dim}")
print(f"向量前10位: {test_embedding[:10]}")

# ====================== 4. 连接 Milvus ======================
milvus_client = MilvusClient(
    uri="./milvus_test_demo.db",
    timeout=30,
    wait_for_ready=True
)
collection_name = "my_1st_rag_collection"

# 清空旧集合
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

# 创建新集合
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",
    consistency_level="Strong"
)

# ====================== 5. 读取 + 正确拆分 md 文件 ======================
text_lines = []
file_path = "mfd.md"

if not os.path.exists(file_path):
    print(f"❌ 错误: 文件 {file_path} 不存在!")
else:
    with open(file_path, "r", encoding="utf-8") as file:
        file_text = file.read()

        # 1. 按标题拆分
        chapter_chunks = re.split(r"#+ ", file_text)
        chapter_chunks = [c.strip() for c in chapter_chunks if c.strip()]

        # 2. 按【法条】二次拆分（最适合民法典）
        for chapter in chapter_chunks:
            law_chunks = re.split(r"(第[一二三四五六七八九十百千万\d]+条)", chapter)
            combined = []
            for i in range(1, len(law_chunks), 2):
                if i + 1 < len(law_chunks):
                    combined.append(law_chunks[i] + law_chunks[i+1])

            # 清理换行、多余空格，过滤太短的片段
            cleaned = [
                re.sub(r"\s+", " ", chunk.strip())
                for chunk in combined
                if chunk.strip() and len(chunk) > 20
            ]
            text_lines.extend(cleaned)

print(f"✅ 拆分完成，共 {len(text_lines)} 段有效内容")

# ====================== 6. 向量化 + 入库 ======================
doc_embeddings = embedding_model.encode_documents(text_lines)
data = []

for i, line in enumerate(tqdm(text_lines, desc="生成向量中")):
    data.append({
        "id": i,
        "vector": doc_embeddings[i],
        "text": line
    })

milvus_client.insert(collection_name=collection_name, data=data)
print("✅ 全部插入 Milvus 向量库成功")

# ====================== 7. 检索 ======================
question = "权利人、利害关系人可以申请查询、复制不动产登记资料吗？中华人民共和国民法典哪条有规定？"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries([question]),
    limit=5,
    search_params={"metric_type": "IP"},
    output_fields=["text"]
)

# 打印检索结果
print("\n" + "="*50)
print("🔍 检索到的最相关内容：")
print("="*50)

retrieved_lines = []
for idx, res in enumerate(search_res[0]):
    text = res["entity"]["text"]
    distance = res["distance"]
    retrieved_lines.append(text)
    print(f"\n【第 {idx+1} 条 | 相似度: {distance:.3f}】")
    print(text[:200] + "..." if len(text) > 200 else text)

# ====================== 8. 大模型回答 ======================
context = "\n".join(retrieved_lines)

SYSTEM_PROMPT = """
你是专业的AI法律助手，只能依据提供的民法典内容回答问题，不能编造。
"""

USER_PROMPT = f"""
请根据下面的民法典内容回答问题。

<context>
{context}
</context>

<question>
{question}
</question>

请直接给出答案，并在最后标注对应的法条。
"""

# 调用大模型
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    temperature=0.1
)

# 输出最终答案
print("\n" + "="*50)
print("📌 最终回答：")
print("="*50)
print(response.choices[0].message.content)

✅ Api key read successfully
向量维度: 768
向量前10位: [-0.02750775 -0.01074684  0.02967544 -0.01778205 -0.01723714 -0.0605397
 -0.0135056   0.06475154  0.01001156  0.01850883]
✅ 拆分完成，共 355 段有效内容


生成向量中: 100%|██████████| 355/355 [00:00<00:00, 337713.30it/s]
I0501 13:02:41.322934    2911 chttp2_transport.cc:1369] unix:/tmp/tmp516jbjxw_milvus_test_demo.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0501 13:02:41.323114    2911 chttp2_transport.cc:1401] unix:/tmp/tmp516jbjxw_milvus_test_demo.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


✅ 全部插入 Milvus 向量库成功

🔍 检索到的最相关内容：

【第 1 条 | 相似度: 0.635】
第五百一十二条** 当事人就有关合同内容约定不明确，依照本法

【第 2 条 | 相似度: 0.635】
第四百七十条** 债务人履行债务或者留置权消灭的，留置权消灭。

【第 3 条 | 相似度: 0.635】
第四百五十六条** 权利质权除适用本节规定外，参照适用本章动产质权的有关规定。

【第 4 条 | 相似度: 0.635】
第三百五十二条** 拾得漂流物、失散的饲养动物或者受到损害的财产，参照适用拾得遗失物的有关规定。

【第 5 条 | 相似度: 0.635】
第二百四十三条** 法律、行政法规对不动产或者动产的归属有特别规定的，依照其规定。

📌 最终回答：
根据提供的民法典内容，无法直接回答该问题，因为所给条款中未涉及不动产登记资料查询、复制的相关规定。建议查阅《中华人民共和国民法典》第二百一十八条。
